# 06 — Drift monitoring and retraining decision
Tính PSI giữa reference window và current window. PSI ≥ 0.25 là tín hiệu xem xét retrain, không phải quyết định tự động promote model.

In [ ]:
from pathlib import Path
import sys, pandas as pd
ROOT = Path.cwd().resolve(); ROOT = ROOT.parent if ROOT.name == 'notebooks' else ROOT
sys.path[:0] = [str(ROOT / 'src'), str(ROOT)]
from cooling_load.config import load_config
from cooling_load.io import read_frame
from cooling_load.monitoring import drift_report
config = load_config(ROOT / 'configs/base.yaml')
features = read_frame(config.path('feature_data'))
features['timestamp'] = pd.to_datetime(features['timestamp'], utc=True)
features = features.sort_values('timestamp')
cutoff = features['timestamp'].quantile(0.7)
reference = features[features.timestamp <= cutoff]
current = features[features.timestamp > cutoff]


In [ ]:
monitored = ['cooling_load_kwh_lag_3','cooling_load_kwh_mean_24','outdoor_temperature_c_lag_3','chilled_water_flow_m3h_lag_3','occupancy_proxy_lag_3']
report = drift_report(reference, current, monitored)
display(report)
print('Retrain candidates:', report.loc[report.status.eq('retrain'), 'feature'].tolist())
